In [ ]:
import requests
import pandas as pd
import os
import re

%pip install requests pandas openpyxl python-dotenv

 

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("CANVAS_BASE_URL"))
print(os.getenv("CANVAS_TOKEN"))

In [ ]:
# 定义API的基本URL和请求头

load_dotenv()

CANVAS_BASE_URL = os.getenv("CANVAS_BASE_URL").rstrip("/")
CANVAS_TOKEN = os.getenv("CANVAS_TOKEN")

API_BASE = f"{CANVAS_BASE_URL}/api/v1"

headers = {
    "Authorization": f"Bearer {CANVAS_TOKEN}",
    "Accept": "application/json"
}

print("Canvas:", CANVAS_BASE_URL)
print("Token loaded:", bool(CANVAS_TOKEN))

Canvas: https://hkust-gz.instructure.com
Token loaded: True


In [ ]:
# 定义获取API需求的函数

def canvas_get(endpoint, params=None):
    """
    Send GET request to Canvas API and return JSON.
    """
    
    url = f"{API_BASE}{endpoint}"
    
    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30
    )
    
    print(f"GET {response.url}")
    print(f"Status: {response.status_code}")
    
    response.raise_for_status()
    
    return response.json()

In [ ]:
# 测试课程数据

course_id = 3308

course = canvas_get(f"/courses/{course_id}")

course["id"], course["name"]

In [ ]:
# 定义2个作业和相应的rubric_id

assignments = {
    11111: {
        "name": "Assignment 1",
        "rubric_id": 1077
    },
    22222: {
        "name": "Assignment 2",
        "rubric_id": 1077
    }
}

In [ ]:
# 获取Rubric信息并建立criterion的字典

rubric_cache = {}

for assignment_id, info in assignments.items():
    
    rubric_id = info["rubric_id"]
    
    rubric = canvas_get(
        f"/courses/{course_id}/rubrics/{rubric_id}"
    )
    
    rubric_cache[rubric_id] = rubric

GET https://hkust-gz.instructure.com/api/v1/courses/3308/rubrics/1077
Status: 200
GET https://hkust-gz.instructure.com/api/v1/courses/3308/rubrics/1077
Status: 200


In [ ]:
# 测试输出Rubric的数据字段

rubric_cache[1077].keys()

dict_keys(['id', 'title', 'context_id', 'context_type', 'points_possible', 'reusable', 'public', 'read_only', 'free_form_criterion_comments', 'hide_score_total', 'data', 'rating_order', 'button_display'])

In [ ]:
# 测试Rubric中data字段的信息

rubric_cache[1077]["data"]

[{'description': 'Asking the right question',
  'long_description': '',
  'points': 18.0,
  'id': '1781512456632',
  'criterion_use_range': False,
  'generated': False,
  'ratings': [{'description': 'Exceeds',
    'long_description': '',
    'points': 18.0,
    'criterion_id': '1781512456632',
    'id': '1'},
   {'description': 'Mastery',
    'long_description': '',
    'points': 13.5,
    'criterion_id': '1781512456632',
    'id': '2'},
   {'description': 'Near',
    'long_description': '',
    'points': 9.0,
    'criterion_id': '1781512456632',
    'id': '3'},
   {'description': 'Below',
    'long_description': '',
    'points': 4.5,
    'criterion_id': '1781512456632',
    'id': '4'},
   {'description': 'No Evidence',
    'long_description': '',
    'points': 0.0,
    'criterion_id': '1781512456632',
    'id': '5'}]},
 {'description': 'Logical structure',
  'long_description': '',
  'points': 18.0,
  'id': '1781512487374',
  'criterion_use_range': False,
  'generated': False,
  'rat

In [18]:
# 建立 criterion mapping

def build_criterion_map(rubric):
    """
    Convert Canvas rubric criteria into:
    
    criterion_id -> {
        name,
        description,
        points
    }
    """
    
    criterion_map = {}
    
    for criterion in rubric.get("data", []):
        
        criterion_id = str(criterion["id"])
        
        criterion_map[criterion_id] = {
            "name": criterion.get("description", criterion_id),
            "long_description": criterion.get("long_description"),
            "points": criterion.get("points")
        }
    
    return criterion_map

In [19]:
criterion_maps = {
    rubric_id: build_criterion_map(rubric)
    for rubric_id, rubric in rubric_cache.items()
}

criterion_maps[1077]

{'1781512456632': {'name': 'Asking the right question',
  'long_description': '',
  'points': 18.0},
 '1781512487374': {'name': 'Logical structure',
  'long_description': '',
  'points': 18.0},
 '1781512498763': {'name': 'Accuracy & comprehensive',
  'long_description': '',
  'points': 18.0},
 '1781512512067': {'name': 'Demonstrate critical thinking and propose new ideas',
  'long_description': '',
  'points': 18.0},
 '1781512526265': {'name': 'Formatting, e.g. figures, tables, text, and proper reference',
  'long_description': '',
  'points': 18.0}}

In [23]:
# 把 criterion 名称变成合法的 Excel 列名

def clean_column_name(name):
    """
    Make rubric criterion name suitable for DataFrame/Excel column names.
    """
    
    name = str(name).strip()
    
    # 空格 → _
    name = re.sub(r"\s+", "_", name)
    
    # 删除特殊字符
    name = re.sub(r"[^\w\u4e00-\u9fff-]", "", name)
    
    return name

In [24]:
def build_column_map(criterion_map):
    
    column_map = {}
    
    for criterion_id, info in criterion_map.items():
        
        criterion_name = clean_column_name(info["name"])
        
        column_map[criterion_id] = {
            "score": f"{criterion_name}_score",
            "comment": f"{criterion_name}_comment"
        }
    
    return column_map

In [25]:
column_maps = {
    rubric_id: build_column_map(criterion_map)
    for rubric_id, criterion_map in criterion_maps.items()
}

column_maps[1077]

{'1781512456632': {'score': 'Asking_the_right_question_score',
  'comment': 'Asking_the_right_question_comment'},
 '1781512487374': {'score': 'Logical_structure_score',
  'comment': 'Logical_structure_comment'},
 '1781512498763': {'score': 'Accuracy__comprehensive_score',
  'comment': 'Accuracy__comprehensive_comment'},
 '1781512512067': {'score': 'Demonstrate_critical_thinking_and_propose_new_ideas_score',
  'comment': 'Demonstrate_critical_thinking_and_propose_new_ideas_comment'},
 '1781512526265': {'score': 'Formatting_eg_figures_tables_text_and_proper_reference_score',
  'comment': 'Formatting_eg_figures_tables_text_and_proper_reference_comment'}}

In [26]:
# 获取 Peer Review 原始数据

peer_raw = {}

for assignment_id, info in assignments.items():
    
    rubric_id = info["rubric_id"]
    
    data = canvas_get(
        f"/courses/{course_id}/rubrics/{rubric_id}",
        params={
            "include[]": "peer_assessments",
            "style": "full"
        }
    )
    
    peer_raw[assignment_id] = data

GET https://hkust-gz.instructure.com/api/v1/courses/3308/rubrics/1077?include%5B%5D=peer_assessments&style=full
Status: 200
GET https://hkust-gz.instructure.com/api/v1/courses/3308/rubrics/1077?include%5B%5D=peer_assessments&style=full
Status: 200


In [29]:
peer_raw[11111]["assessments"][:2]

[{'id': 3799,
  'rubric_id': 1077,
  'rubric_association_id': 2051,
  'score': 49.5,
  'artifact_type': 'Submission',
  'artifact_id': 744176,
  'artifact_attempt': 2,
  'assessment_type': 'peer_review',
  'assessor_id': 10716,
  'data': [{'id': '2',
    'points': 13.5,
    'criterion_id': '1781512456632',
    'learning_outcome_id': None,
    'description': 'Mastery',
    'comments_enabled': True,
    'comments': 'Justification for score ：\nThe introduction clearly raises the core research question: “This raises a critical question: Whether enhanced weathering supports an elevated level of net carbon sink efficiency”. The whole paper splits analysis into terrestrial and marine EW, matching the title’s research scope, and the abstract restates this framework consistently.\nHowever, the text never sets clear comparative expectations for land and marine EW at the end of introduction, making the research goal slightly unfocused, so it cannot reach Exceeds standard\n\nConstructive Feedback：

In [31]:
# 获取peer review -> student的对应关系

peer_review_map = {}

for assignment_id in assignments:
    
    peer_reviews = canvas_get(
        f"/courses/{course_id}/assignments/{assignment_id}/peer_reviews"
    )
    
    peer_review_map[assignment_id] = peer_reviews

GET https://hkust-gz.instructure.com/api/v1/courses/3308/assignments/11111/peer_reviews
Status: 404


HTTPError: 404 Client Error: Not Found for url: https://hkust-gz.instructure.com/api/v1/courses/3308/assignments/11111/peer_reviews

In [32]:
peer_review_map[11111][:2]

KeyError: 11111

In [30]:
artifact_to_student = {}

for assignment_id, peer_reviews in peer_review_map.items():
    
    artifact_to_student[assignment_id] = {}
    
    for review in peer_reviews:
        
        artifact_id = review.get("asset_id")
        student_id = review.get("user_id")
        
        artifact_to_student[assignment_id][artifact_id] = student_id

NameError: name 'peer_review_map' is not defined